# Rainier: ComCat recovery, TM versus ML, and DAS versus five stations

This notebook regenerates the revised **single figure with panels a), b), and c)** directly from the eight supplied input CSVs. It recalculates temporal matches; it does not load fixed figure counts.

- **a)** ComCat recovery by TM and ML, with the full-network ML result as context.
- **b)** Direct TM–ML comparison, separately for DAS and stations.
- **c)** Direct DAS–five-station comparison using ML.

Defaults are **±7 seconds** and a **25-km ComCat radius for panel a only**. Panels b–c use full selected catalogs in the common calendar interval, without a ComCat or location restriction. Change the controls in section 1, then choose **Run All**. Colors, titles and layout are editable in section 8.

Place your local CSV inputs under `text-file/ml-tm-das-stations/inputs/` using `prepare_inputs.py`; research CSVs are not distributed in GitHub. Calculation dependencies: `numpy`, `scipy`, `matplotlib`; Jupyter/IPython displays the tables and figure. If needed: `%pip install numpy scipy matplotlib`.

ML picks are PhaseNet-DAS for DAS and ELEP for stations, with GaMMA association. The station-TM production subset is unverified. TM peak timestamps and ML origins are compared as recorded; DAS clock drift is uncorrected. Unmatched records do not establish new or distinct earthquakes.


## 1. Configuration

The eight input CSVs are local research data. Use `prepare_inputs.py` to place them under `text-file/ml-tm-das-stations/inputs/` before running. The packages required for calculations are `numpy`, `scipy` and `matplotlib`; Jupyter/IPython displays the tables and figures. No pandas dependency is needed.

If packages are missing, install them in your notebook environment with `%pip install numpy scipy matplotlib` and restart the kernel before running all cells.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import csv, hashlib, html, json, math, os, platform, tempfile
import numpy as np
from scipy.optimize import linear_sum_assignment
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import maximum_bipartite_matching

os.environ.setdefault('MPLCONFIGDIR', str(Path(tempfile.gettempdir()) / 'rainier_notebook_mpl'))
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from IPython.display import display, HTML, Image, Markdown

# Locate the repository from either its root or this notebook's directory.
search_roots = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next((p for p in search_roots
                  if (p / 'text-file/ml-tm-das-stations/inputs/gamma_das_candidates.csv').is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Run from the repository root or notebooks/ml-tm-das-stations; keep the shared text-file inputs in this checkout.')
BASE_DIR = REPO_ROOT / 'notebooks/ml-tm-das-stations'
INPUT_DIR = REPO_ROOT / 'text-file/ml-tm-das-stations/inputs'
OUTPUT_DIR = BASE_DIR / 'results/summary'
FIGURE_DIR = OUTPUT_DIR / 'figures'
MATCH_DIR = OUTPUT_DIR / 'matched_pairs'

TOLERANCE_SECONDS = 7.0
SENSITIVITY_TOLERANCES = [1, 2, 3, 5, 7, 10, 15, 20, 30]
START_UTC = '2023-08-23T21:00:00Z'       # inclusive
END_UTC = '2023-11-22T00:00:00Z'         # exclusive
TM_DAS_CC_MIN = 0.07                     # strict CC > threshold
INCLUDE_UNCERTAIN_DAS_LINKS = True       # reproduces the agreed 478-group selection
EXPORT_FILES = True

# ComCat reference uses the association's horizontal bounds, not a 25-km circle.
COMCAT_LATITUDE = (45.0, 48.0)
COMCAT_LONGITUDE = (-123.0, -120.0)
COMCAT_EVENT_TYPES = ('earthquake',)       # excludes identified explosions/quarry blasts
COMCAT_DEPTH_KM = None                    # e.g. (0.0, 40.0) to impose the nominal depth range
LOCAL_REFERENCE_RADIUS_KM = 25.0          # main local benchmark; regional box is separate
LOCAL_REFERENCE_CENTER = (46.879967, -121.726906)

FILES = {
    'DAS candidates': INPUT_DIR / 'gamma_das_candidates.csv',
    'Five stations': INPUT_DIR / 'gamma_five_station_events.csv',
    'Full network': INPUT_DIR / 'gamma_all_station_events.csv',
    'TM-DAS': INPUT_DIR / 'tm_das_detections.csv',
    'TM-stations': INPUT_DIR / 'tm_station_detections.csv',
    'Original confirmed groups': INPUT_DIR / 'gamma_original_confirmed_groups.csv',
    'Review-to-new-run mapping': INPUT_DIR / 'gamma_confirmed_group_to_new_run.csv',
    'ComCat regional source': INPUT_DIR / 'comcat_regional_raw.csv',
}
assert TOLERANCE_SECONDS > 0 and 0 <= TM_DAS_CC_MIN <= 1
for folder in [OUTPUT_DIR, FIGURE_DIR, MATCH_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
missing_files = [str(p) for p in FILES.values() if not p.is_file()]
if missing_files:
    raise FileNotFoundError('Missing input CSVs. Edit BASE_DIR/FILES: ' + ', '.join(missing_files))
print(f'Input folder: {INPUT_DIR}')
print(f'Matching tolerance: ±{TOLERANCE_SECONDS:g} s; TM-DAS CC > {TM_DAS_CC_MIN:g}')
print(f'UTC interval: [{START_UTC}, {END_UTC})')


## 2. Load the CSVs and show their provenance

The original files are never edited. Each row in the station event tables is one associated event record, not one pick. `gamma_das_candidates.csv` contains all 942 new-run candidates; the review mapping below selects the previously confirmed groups.


In [ ]:
def read_csv(path):
    with Path(path).open(newline='', encoding='utf-8-sig') as handle:
        reader = csv.DictReader(handle)
        if reader.fieldnames is None:
            raise ValueError(f'CSV has no header: {path}')
        return list(reader)

def show_table(rows, columns=None, max_rows=None):
    if not rows:
        display(HTML('<p>No records.</p>'))
        return
    columns = list(columns or rows[0])
    visible = rows if max_rows is None else rows[:max_rows]
    def cell(value):
        if isinstance(value, float):
            return f'{value:,.2f}'
        return html.escape(str(value))
    head = ''.join(f'<th style="padding:7px;text-align:left">{html.escape(str(k))}</th>' for k in columns)
    body = ''.join('<tr>' + ''.join(f'<td style="padding:7px;border-bottom:1px solid #dde3e9">{cell(row.get(k, ""))}</td>' for k in columns) + '</tr>' for row in visible)
    display(HTML('<table style="border-collapse:collapse;font-family:Arial;font-size:13px"><thead style="background:#eaf0f5"><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))
    if len(visible) < len(rows):
        print(f'Showing {len(visible)} of {len(rows)} records.')

def write_csv(path, rows, columns=None):
    rows = list(rows)
    columns = list(columns or (rows[0].keys() if rows else []))
    with Path(path).open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=columns, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)

RAW = {name: read_csv(path) for name, path in FILES.items()}
provenance = [{'Input': name, 'CSV file': path.name, 'Rows': len(RAW[name]),
               'SHA256': hashlib.sha256(path.read_bytes()).hexdigest()}
              for name, path in FILES.items()]
show_table(provenance, ['Input', 'CSV file', 'Rows'])


## 3. Reproduce the reviewed DAS selection

The original review contains **483 consolidated, confirmed groups**. The supplied mapping links **478** of those groups to distinct records in the new GaMMA run; **5** remain unmatched. Of the 478 links, 138 are flagged uncertain and are retained by default, as in the comparison already discussed.

The mapping CSV is a saved result of the earlier catalog-correspondence analysis; it does not claim a new waveform review. Changing the input event catalog may require a new mapping. Matching DAS to stations below is a separate calculation with its own tolerance.


In [ ]:
def parse_utc(text):
    value = datetime.fromisoformat(str(text).replace('Z', '+00:00'))
    return value.replace(tzinfo=timezone.utc) if value.tzinfo is None else value.astimezone(timezone.utc)

start = parse_utc(START_UTC)
end = parse_utc(END_UTC)
assert start < end

groups = RAW['Original confirmed groups']
mapping = RAW['Review-to-new-run mapping']
group_ids = {r['event_id'] for r in groups}
assert len(group_ids) == len(groups), 'Duplicate historical group IDs'
assert {r['historical_event_id'] for r in mapping} == group_ids
assert len(mapping) == len(group_ids), 'Duplicate mapping rows'
assert all(r['confirm'].lower() == 'yes' for r in groups)
assert all(r['historical_confirm'].lower() == 'yes' for r in mapping)

new_by_id = {r['event_index']: r for r in RAW['DAS candidates']}
assert len(new_by_id) == len(RAW['DAS candidates']), 'Duplicate new-run event IDs'
linked = [r for r in mapping if r['mapping_status'] == 'linked']
unlinked = [r for r in mapping if r['mapping_status'] == 'unmatched']
assert len(linked) + len(unlinked) == len(groups)
assert len({r['new_event_index'] for r in linked}) == len(linked), 'A new record was reused'
assert all(r['new_event_index'] in new_by_id for r in linked)
uncertain_links = [r for r in linked if r['historical_link_uncertain'].lower() == 'true']
selected_links = [r for r in linked if INCLUDE_UNCERTAIN_DAS_LINKS or r['historical_link_uncertain'].lower() != 'true']

review_counts = [
    {'Selection step': 'All new GaMMA-DAS candidates', 'Count': len(RAW['DAS candidates'])},
    {'Selection step': 'Original confirmed/consolidated groups', 'Count': len(groups)},
    {'Selection step': 'Groups linked to new-run records', 'Count': len(linked)},
    {'Selection step': 'Groups without an assigned new-run record', 'Count': len(unlinked)},
    {'Selection step': 'Uncertain links among linked groups', 'Count': len(uncertain_links)},
    {'Selection step': 'Linked groups retained by the current setting', 'Count': len(selected_links)},
]
show_table(review_counts)
if unlinked:
    print('Historical confirmed groups without a new-run counterpart:')
    show_table(unlinked, ['historical_event_id', 'historical_representative_idx', 'match_basis'])


## 4. Build the five catalogs used in the comparison

- **ML-DAS:** one new-run representative per selected reviewed DAS group.
- **ML-five-stations:** all events in the five-station GaMMA CSV, accepted as valid by the study assumption.
- **ML-full-network:** the full GaMMA station catalog, reported separately for context.
- **TM-DAS:** the supplied DAS detections after the agreed `CC > 0.07` selection.
- **TM-stations:** all supplied station-TM detections. Its station configuration is not established by the CSV, so it is **not relabeled as five-station TM**.

The calendar window is the common interval used in the written comparison. It is not an uptime correction. A missing temporal match does not prove a new or distinct physical earthquake.


In [ ]:
def ml_record(row, link=None):
    stamp = parse_utc(row['time'])
    return {'id': str(row['event_index']), 'time_utc': stamp.isoformat(),
            'time': stamp, 'seconds': stamp.timestamp(),
            'latitude': float(row['latitude']), 'longitude': float(row['longitude']),
            'depth_km': float(row['depth_km']), 'picks': int(row['num_picks']),
            'historical_group_id': link['historical_event_id'] if link else '',
            'historical_link_uncertain': bool(link and link['historical_link_uncertain'].lower() == 'true')}

def tm_records(rows, time_column, cc_column):
    result = []
    for row_number, row in enumerate(rows, start=2):
        stamp = parse_utc(row[time_column])
        result.append({'id': str(row_number), 'time_utc': stamp.isoformat(),
                       'time': stamp, 'seconds': stamp.timestamp(),
                       'CC': float(row[cc_column]),
                       'template_source': row.get('template_id', row.get('source_file', '')),
                       'historical_link_uncertain': False})
    return result

def within_window(records):
    records = [r for r in records if start <= r['time'] < end]
    records.sort(key=lambda r: (r['seconds'], r['id']))
    assert len({r['id'] for r in records}) == len(records)
    return records

before_window = {
    'ML-DAS': [ml_record(new_by_id[r['new_event_index']], r) for r in selected_links],
    'ML-five-stations': [ml_record(r) for r in RAW['Five stations']],
    'ML-full-network': [ml_record(r) for r in RAW['Full network']],
    'TM-DAS': [r for r in tm_records(RAW['TM-DAS'], 'UTC', 'CC') if r['CC'] > TM_DAS_CC_MIN],
    'TM-stations': tm_records(RAW['TM-stations'], 'event_time_utc', 'cc'),
}
CATALOGS = {name: within_window(rows) for name, rows in before_window.items()}
LABELS = {'ML-DAS': 'DAS: ML + GaMMA (reviewed)',
          'ML-five-stations': 'Five stations: ML + GaMMA (accepted)',
          'ML-full-network': 'Full network: ML + GaMMA (context)',
          'TM-DAS': 'DAS: template matching',
          'TM-stations': 'Stations: template matching (subset unverified)'}
totals = [{'Catalog': LABELS[name], 'Before calendar filter': len(rows),
           'Used in comparison': len(CATALOGS[name]), 'Outside interval': len(rows)-len(CATALOGS[name])}
          for name, rows in before_window.items()]
show_table(totals)
print('Full-network ML is context; the main station ML comparison uses the five-station catalog.')


## 5. One-to-one temporal matching

For each pair of catalogs, all candidate correspondences within the time tolerance are considered. The assignment maximizes the number of pairs and then minimizes their summed absolute time differences. Each record can appear at most once within that comparison. The maximum number of pairs is independently checked with a bipartite-graph matching calculation.

`Left only = left total − shared` and `Right only = right total − shared`. We also flag pairs with competing time candidates and uncertain historical DAS links. All comparisons are recalculated from the CSV-loaded records, not copied from fixed summary counts.


In [ ]:
def temporal_match(left_name, right_name, tolerance=TOLERANCE_SECONDS):
    left, right = CATALOGS[left_name], CATALOGS[right_name]
    if not left or not right:
        indices = []
        delta = np.empty((len(left), len(right)))
        eligible = np.zeros(delta.shape, dtype=bool)
    else:
        delta = np.array([r['seconds'] for r in right])[None, :] - np.array([r['seconds'] for r in left])[:, None]
        eligible = np.abs(delta) <= tolerance
        ii, jj = linear_sum_assignment(np.where(eligible, np.abs(delta)/(tolerance+1), 1e6))
        indices = [(int(i), int(j)) for i, j in zip(ii, jj) if eligible[i, j]]
        maximum = maximum_bipartite_matching(csr_matrix(eligible), perm_type='column')
        assert len(indices) == int((maximum >= 0).sum()), 'Assignment did not maximize pair count'
    used_left = {i for i, j in indices}
    used_right = {j for i, j in indices}
    assert len(used_left) == len(used_right) == len(indices)
    pairs = []
    for i, j in indices:
        a, b = left[i], right[j]
        pairs.append({'left_id': a['id'], 'right_id': b['id'],
                      'left_time_utc': a['time_utc'], 'right_time_utc': b['time_utc'],
                      'right_minus_left_s': float(delta[i, j]),
                      'left_historical_group_id': a.get('historical_group_id', ''),
                      'right_historical_group_id': b.get('historical_group_id', ''),
                      'historical_DAS_link_uncertain': bool(a.get('historical_link_uncertain') or b.get('historical_link_uncertain')),
                      'competing_time_candidates': bool(eligible[i].sum() > 1 or eligible[:, j].sum() > 1)})
    assert all(abs(p['right_minus_left_s']) <= tolerance for p in pairs)
    return {'left': left_name, 'right': right_name, 'tolerance_s': float(tolerance),
            'left_total': len(left), 'right_total': len(right), 'shared': len(pairs),
            'left_only': len(left)-len(pairs), 'right_only': len(right)-len(pairs),
            'pairs_with_competing_candidates': sum(p['competing_time_candidates'] for p in pairs),
            'pairs_with_uncertain_DAS_link': sum(p['historical_DAS_link_uncertain'] for p in pairs),
            'pairs': pairs,
            'left_unmatched': [r for i, r in enumerate(left) if i not in used_left],
            'right_unmatched': [r for j, r in enumerate(right) if j not in used_right]}

COMPARISONS = {
    'DAS: ML versus TM': ('ML-DAS', 'TM-DAS'),
    'Five-station ML versus station TM': ('ML-five-stations', 'TM-stations'),
    'Full-network ML versus station TM (context)': ('ML-full-network', 'TM-stations'),
    'ML: DAS versus five stations': ('ML-DAS', 'ML-five-stations'),
    'ML: DAS versus full network (context)': ('ML-DAS', 'ML-full-network'),
    'TM: DAS versus stations': ('TM-DAS', 'TM-stations'),
}
MATCHES = {name: temporal_match(*catalogs) for name, catalogs in COMPARISONS.items()}
summary_rows = [{'Comparison': name, 'Left total': r['left_total'], 'Shared': r['shared'],
                 'Left only': r['left_only'], 'Right total': r['right_total'], 'Right only': r['right_only'],
                 'Competing time candidates': r['pairs_with_competing_candidates']}
                for name, r in MATCHES.items()]
show_table(summary_rows)
for r in MATCHES.values():
    assert r['left_total'] == r['shared'] + r['left_only']
    assert r['right_total'] == r['shared'] + r['right_only']


## 6. Write down the TM/ML accounting

This table uses **complete selected catalogs**, without a ComCat restriction. “ML only” means no assigned TM counterpart within the selected tolerance; it does not mean “new relative to a published catalog.” The five-station ML and station-TM comparison is not a controlled five-station-versus-five-station test because the TM production station subset is unknown.


In [ ]:
accounting = []
for label, key in [('DAS', 'DAS: ML versus TM'),
                   ('Five-station ML versus existing station TM', 'Five-station ML versus station TM'),
                   ('Full-network ML (context)', 'Full-network ML versus station TM (context)')]:
    r = MATCHES[key]
    accounting.append({'System': label, 'ML total': r['left_total'], 'TM total': r['right_total'],
                       'Shared': r['shared'], 'ML only': r['left_only'], 'TM only': r['right_only']})
show_table(accounting)
for row in accounting:
    print(f"{row['System']}: ML {row['ML total']:,} = {row['Shared']:,} shared + {row['ML only']:,} ML only")
    print(f"{' '*len(row['System'])}  TM {row['TM total']:,} = {row['Shared']:,} shared + {row['TM only']:,} TM only")


## 7. Select the local ComCat reference for panel a

The saved regional ComCat source is filtered by the calendar, event type and horizontal bounds. The local reference is then selected using ComCat epicenters within `LOCAL_REFERENCE_RADIUS_KM` of the original Rainier query center. All selected ML and TM records remain eligible for temporal matching. No equivalent radius filter is applied to panels b or c.


In [ ]:
lat_min, lat_max = COMCAT_LATITUDE
lon_min, lon_max = COMCAT_LONGITUDE
assert lat_min < lat_max and lon_min < lon_max
regional_source = RAW['ComCat regional source']

def haversine_km(lat1,lon1,lat2,lon2):
    a,b = np.deg2rad([lat1,lat2]); dlon=np.deg2rad(lon2-lon1)
    h=np.sin((b-a)/2)**2+np.cos(a)*np.cos(b)*np.sin(dlon/2)**2
    return float(6371.0*2*np.arcsin(np.sqrt(np.clip(h,0,1))))

maximum_corner_distance=max(haversine_km(46.879967,-121.726906,la,lo)
                            for la in COMCAT_LATITUDE for lo in COMCAT_LONGITUDE)
if maximum_corner_distance >= 299:
    raise ValueError('The edited rectangle may extend outside the saved 300-km source catalog. Supply a wider ComCat input before comparing.')

regional_box=[]
for row in regional_source:
    stamp=parse_utc(row['time'])
    if (start <= stamp < end and lat_min <= float(row['latitude']) <= lat_max
            and lon_min <= float(row['longitude']) <= lon_max):
        regional_box.append(row)
assert len({r['id'] for r in regional_box})==len(regional_box)
type_counts=Counter(r['type'] for r in regional_box)
regional_typed=[r for r in regional_box if COMCAT_EVENT_TYPES is None or r['type'] in COMCAT_EVENT_TYPES]
regional_selected=regional_typed
if COMCAT_DEPTH_KM is not None:
    zmin,zmax=COMCAT_DEPTH_KM
    assert zmin <= zmax
    regional_selected=[r for r in regional_typed if zmin <= float(r['depth']) <= zmax]

reference_records=[]
for row in regional_selected:
    stamp=parse_utc(row['time'])
    reference_records.append({'id':row['id'],'time_utc':stamp.isoformat(),'time':stamp,'seconds':stamp.timestamp(),
                              'latitude':float(row['latitude']),'longitude':float(row['longitude']),
                              'depth_km':float(row['depth']),'magnitude':float(row['mag']) if row['mag'] else None,
                              'event_type':row['type'],'historical_link_uncertain':False})
CATALOGS['ComCat regional']=within_window(reference_records)
reference_n=len(CATALOGS['ComCat regional'])
if reference_n == 0:
    raise ValueError('No ComCat records remain after the configured filters.')


local_reference_records = [r for r in CATALOGS['ComCat regional']
                          if haversine_km(*LOCAL_REFERENCE_CENTER, r['latitude'], r['longitude']) <= LOCAL_REFERENCE_RADIUS_KM]
if not local_reference_records:
    raise ValueError('No ComCat events remain inside the local reference radius.')
CATALOGS['ComCat local'] = local_reference_records
LOCAL_MATCHES = {name: temporal_match('ComCat local', name) for name in before_window}
LOCAL_SUMMARY = [{'Catalog':LABELS[name], 'Local ComCat reference':len(local_reference_records),
                  'Counterparts':r['shared'], 'Local ComCat without counterpart':r['left_only'],
                  'Percent with counterpart':100*r['shared']/len(local_reference_records)}
                 for name,r in LOCAL_MATCHES.items()]
show_table(LOCAL_SUMMARY)

from datetime import timedelta

# Adapt the calculated tables to the plotting cell.
local = {r['Catalog']:r for r in LOCAL_SUMMARY}
totals = {LABELS[name]:len(CATALOGS[name]) for name in before_window}
method_rows = {r['System']:r for r in accounting}
direct = next(r for r in summary_rows if r['Comparison']=='ML: DAS versus five stations')
N_REFERENCE = len(local_reference_records)
OUT = FIGURE_DIR
date_label = start.strftime('%-d %B')+'–'+(end-timedelta(microseconds=1)).strftime('%-d %B %Y')


## 8. Draw the summary figure

Edit the colors, font sizes, text and axes positions below. The numerical labels are calculated above. PNG, PDF and editable SVG files are saved to `results/figures/`; the figure is also displayed inline.


In [ ]:
INK = '#253645'
MUTED = '#536575'
DAS = '#79549C'
FIVE = '#397FA2'
FULL = '#697987'
MATCHED = '#258F87'
UNMATCHED = '#CA7741'
PALE = '#D7DDE1'
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 11,
                     'text.color': INK, 'axes.labelcolor': INK,
                     'axes.edgecolor': '#A8B3BC', 'xtick.color': INK, 'ytick.color': INK,
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.spines.left': False, 'pdf.fonttype': 42,
                     'svg.fonttype': 'none', 'figure.facecolor': 'white',
                     'savefig.facecolor': 'white'})


def save(fig, stem):
    for ext in ['png', 'pdf', 'svg']:
        fig.savefig(OUT / f'{stem}.{ext}', dpi=240)
    plt.close(fig)


fig = plt.figure(figsize=(19.4, 8.8))
a = fig.add_axes([.125, .32, .215, .43])
b = fig.add_axes([.485, .32, .205, .43])
c = fig.add_axes([.835, .32, .14, .43])
fig.text(.033, .947, 'DAS and seismic stations: detection comparison', fontsize=23, fontweight='bold')
fig.text(.033, .900, f'{date_label}   |   GaMMA association   |   One-to-one temporal matching within ±{TOLERANCE_SECONDS:g} s', fontsize=12, color=MUTED)
fig.text(.033, .865, 'ML picks: PhaseNet-DAS for DAS; ELEP for seismic stations. TM = template matching.', fontsize=11, color=MUTED)

for x, title, subtitle in [
    (.033, 'a) Recovery of local\nComCat earthquakes', f'Reference: {N_REFERENCE} earthquakes within {LOCAL_REFERENCE_RADIUS_KM:g} km'),
    (.393, 'b) Template matching\nversus machine learning', 'Full catalogs; no ComCat or radius filter'),
    (.758, 'c) DAS versus five stations\nusing machine learning', 'Full catalogs; no ComCat or radius filter'),
]:
    fig.text(x, .805, title, fontsize=14.5, fontweight='bold', va='bottom', linespacing=1.3)
    fig.text(x, .765, subtitle, fontsize=10.5, color=MUTED)

catalogs = [
    ('DAS: template matching', 'DAS · TM', DAS, True),
    ('DAS: ML + GaMMA (reviewed)', 'DAS · ML', DAS, False),
    ('Stations: template matching (subset unverified)', 'Stations · TM*', FIVE, True),
    ('Five stations: ML + GaMMA (accepted)', '5 stations · ML', FIVE, False),
    ('Full network: ML + GaMMA (context)', 'Full network · ML', FULL, False),
]
for i, (key, label, color, tm) in enumerate(catalogs):
    row = local[key]
    n = int(row['Counterparts'])
    assert int(row['Local ComCat reference']) == N_REFERENCE
    a.barh(i, 100*n/N_REFERENCE, height=.52, color=color, alpha=.42 if tm else 1,
           edgecolor=color if tm else 'none', hatch='///' if tm else None)
    a.text(100*n/N_REFERENCE+1.8, i, f'{n} ({100*n/N_REFERENCE:.1f}%)', va='center', fontsize=10.5)
a.set(yticks=range(5), yticklabels=[f'{label}\nN = {totals[key]:,}' for key, label, _, _ in catalogs],
      xlim=(0, 123), xticks=[0, 25, 50, 75, 100], ylim=(4.65, -.65),
      xlabel='ComCat earthquakes recovered (%)')
a.tick_params(axis='y', length=0, pad=9, labelsize=10.5)
a.set_axisbelow(True)
a.grid(axis='x', color='#E5E9ED', lw=.7)
fig.text(.033, .245, 'N is the full selected catalog size,\nnot the number of ComCat matches.', fontsize=10.5, color=MUTED, va='top')

accounting = []
for i, key in enumerate(['DAS', 'Five-station ML versus existing station TM']):
    row = {k:int(v) if k != 'System' else v for k,v in method_rows[key].items()}
    assert row['TM only']+row['Shared'] == row['TM total']
    assert row['ML only']+row['Shared'] == row['ML total']
    offset = 0
    for field, color in [('TM only',UNMATCHED),('Shared',MATCHED),('ML only','#487DA7')]:
        n = row[field]
        b.barh(i, n, left=offset, color=color, height=.50)
        b.text(offset+n/2, i, str(n), ha='center', va='center', color='white', fontsize=10, fontweight='bold')
        offset += n
    b.text(offset+10, i, f'{offset} total', va='center', fontsize=10.5)
    accounting.append(row)
b.set(yticks=range(2), yticklabels=['DAS', 'Stations*'], ylim=(1.6, -.6),
      xlim=(0, max(680, 1.3*max(int(r['TM only'])+int(r['Shared'])+int(r['ML only']) for r in method_rows.values() if r['System']!='Full-network ML (context)'))) , xticks=[0, 200, 400, 600], xlabel='TM–ML union (matched pairs counted once)')
b.tick_params(axis='y', length=0, pad=10)
b.set_axisbelow(True)
b.grid(axis='x', color='#E5E9ED', lw=.7)
b.legend(handles=[Patch(color=UNMATCHED, label='TM only'),
                  Patch(color=MATCHED, label='Both TM and ML'),
                  Patch(color='#487DA7', label='ML only')],
         loc='upper left', bbox_to_anchor=(-.02, -.18), frameon=False, fontsize=10.5)

shared = int(direct['Shared'])
for i, (n, color) in enumerate([(int(direct['Left only']), DAS), (int(direct['Right only']), FIVE)]):
    c.barh(i, shared, color=MATCHED, height=.52)
    c.barh(i, n, left=shared, color=color, height=.52)
    c.text(shared/2, i, str(shared), va='center', ha='center', fontsize=10.5, color='white', fontweight='bold')
    c.text(shared+n/2, i, str(n), va='center', ha='center', fontsize=10.5, color='white', fontweight='bold')
c.set(yticks=range(2), yticklabels=[f'DAS ML\nN = {direct["Left total"]:,}', f'5-station ML\nN = {direct["Right total"]:,}'], ylim=(1.6, -.6),
      xlim=(0, max(500, 50*math.ceil(max(direct['Left total'],direct['Right total'])/50))), xticks=[0, 100, 200, 300, 400, 500], xlabel='ML catalog records')
c.tick_params(axis='y', length=0, pad=9, labelsize=10.5)
c.set_axisbelow(True)
c.grid(axis='x', color='#E5E9ED', lw=.7)
c.legend(handles=[Patch(color=MATCHED,label=f'Shared: {shared} pairs'),Patch(color=DAS,label='DAS only'),Patch(color=FIVE,label='5-station only')],
         loc='upper left',bbox_to_anchor=(-.02,-.18),frameon=False,fontsize=10.5)

fig.text(.033, .133, f'DAS ML: {len(RAW["DAS candidates"])} association candidates; {len(CATALOGS["ML-DAS"])} records linked to previously reviewed and consolidated earthquakes. Five-station ML records are accepted by study assumption.', fontsize=10.5, color=MUTED)
fig.text(.033, .098, f'*Station ML in (b) uses the five nearby stations ({len(CATALOGS["ML-five-stations"])} records); the TM production station subset ({len(CATALOGS["TM-stations"])} detections) is unverified.', fontsize=10.5, color=MUTED)
fig.text(.033, .063, f'DAS review mapping retains {sum(r["historical_link_uncertain"] for r in CATALOGS["ML-DAS"])} uncertain links. TM peak times are used as recorded; DAS clock drift is uncorrected. Unmatched records are not proven unique earthquakes.', fontsize=10.5, color=MUTED)
save(fig, f'Rainier_DAS_TM_ML_summary_{TOLERANCE_SECONDS:g}s')
display(Image(filename=str(OUT/f'Rainier_DAS_TM_ML_summary_{TOLERANCE_SECONDS:g}s.png')))


## 9. Export the calculated tables and caption


In [ ]:
write_csv(OUTPUT_DIR/'local_comcat_counterpart_summary.csv', LOCAL_SUMMARY)
write_csv(OUTPUT_DIR/'catalog_totals.csv', [{'Catalog':LABELS[k],'Selected records':len(CATALOGS[k])} for k in before_window])
write_csv(OUTPUT_DIR/'comparison_summary.csv', summary_rows)
write_csv(OUTPUT_DIR/'tm_ml_accounting.csv', list(method_rows.values()))
for index,(name,r) in enumerate(MATCHES.items(), start=1):
    write_csv(MATCH_DIR/f'{index:02d}_matched_pairs.csv', r['pairs'])
for name,r in LOCAL_MATCHES.items():
    write_csv(MATCH_DIR/f'local_reference_{name}_pairs.csv', r['pairs'])

caption = (f'DAS and seismic-station detection correspondence over {start.isoformat()} to {end.isoformat()} (end exclusive). '
           f'All pairwise temporal assignments are one-to-one within ±{TOLERANCE_SECONDS:g} seconds. '
           f'(a) Recovery of {N_REFERENCE} ComCat earthquakes within {LOCAL_REFERENCE_RADIUS_KM:g} km of '
           f'{LOCAL_REFERENCE_CENTER[0]} N, {abs(LOCAL_REFERENCE_CENTER[1])} W. N gives the full selected method catalog, not just ComCat matches. '
           '(b) Direct TM–ML comparison using complete selected catalogs, with a matched pair counted once in the union. '
           '(c) Direct DAS–five-station ML comparison. Shared pairs appear in both bars because each bar represents one input catalog. '
           'Panels b–c do not require a ComCat counterpart and do not restrict epicenters. '
           'DAS ML uses reviewed/consolidated historical groups linked to new GaMMA records; uncertain links are retained according to the configuration. '
           'Station ML events are accepted by study assumption; the station-TM production subset is unverified as the five-station array. '
           'TM peak times and ML origin times are used as recorded, without DAS clock-drift correction. '
           'Temporal nonmatches do not establish distinct or previously uncatalogued earthquakes; common dates do not establish equal uptime.\n')
(OUTPUT_DIR/'figure_caption.txt').write_text(caption)
summary = {'configuration':{'tolerance_seconds':TOLERANCE_SECONDS,'radius_km_panel_a':LOCAL_REFERENCE_RADIUS_KM,
                           'start_inclusive':START_UTC,'end_exclusive':END_UTC},
           'local_comcat_recovery':LOCAL_SUMMARY,'tm_ml_accounting':list(method_rows.values()),
           'direct_DAS_five_station_comparison':direct,'input_provenance':provenance}
(OUTPUT_DIR/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
print('Saved figure:', FIGURE_DIR)
